# ДЗ 1

## Бэггинг и бустинг
Для датасета creditcardfraud (мошенничество с кредитными картами, 0 - нет мошенничества, 1 - есть)
- подобрать параметры с помощью GridSearchCV
- вывести и выбрать наилучшие
- обучить бэггинг и бустинг
- сравнить модели (вывести информацию по метрикам)

Для выполнения можно воспользоваться кодом с семинара. Код для загрузки датасета ниже:

In [ ]:
import pandas as pd
df = pd.read_csv('./creditcard.csv')

In [ ]:
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0.0
1,0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0.0
2,1,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0.0
3,1,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0.0
4,2,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0.0


In [ ]:

print("Размер таблицы:", df.shape)
print("\nКоличество пропусков:", df.isnull().sum().sum())
print("\nРаспределение классов:")
print(df["Class"].value_counts())

Размер таблицы: (49610, 31)

Количество пропусков: 27

Распределение классов:
Class
0.0    49461
1.0      148
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

# Удаляем строки с пропущенными значениями
df = df.dropna().copy()

# X — данные для обучения
# y — правильные ответы (0 или 1)
X = df.drop(columns=["Class"])
y = df["Class"].astype(int)

# Делим данные на обучающую и тестовую части
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Обучающая выборка:", X_train.shape)
print("Тестовая выборка:", X_test.shape)
print("Мошеннических операций в обучающей выборке:", y_train.sum())
print("Мошеннических операций в тестовой выборке:", y_test.sum())


Обучающая выборка: (39687, 30)
Тестовая выборка: (9922, 30)
Мошеннических операций в обучающей выборке: 118
Мошеннических операций в тестовой выборке: 30


In [ ]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

# Создаём модель бэггинга
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ),
    random_state=42,
    n_jobs=-1
)

# Параметры для подбора
param_grid = {
    "n_estimators": [10, 30],
    "max_samples": [0.7, 1.0],
    "estimator__max_depth": [5, 10]
}

# Подбор лучших параметров
grid_bagging = GridSearchCV(
    estimator=bagging,
    param_grid=param_grid,
    scoring="f1",
    cv=3,
    n_jobs=2,
    verbose=1
)

grid_bagging.fit(X_train, y_train)

print("Лучшие параметры:", grid_bagging.best_params_)
print("Лучший F1-score:", grid_bagging.best_score_)

Fitting 3 folds for each of 8 candidates, totalling 24 fits
Лучшие параметры: {'estimator__max_depth': 10, 'max_samples': 0.7, 'n_estimators': 30}
Лучший F1-score: 0.8810389610389611


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

# Создаём модель бустинга
boosting = GradientBoostingClassifier(
    random_state=42
)

# Параметры для подбора
param_grid_boosting = {
    "n_estimators": [50, 100],
    "learning_rate": [0.05, 0.1],
    "max_depth": [2, 3]
}

# Подбираем лучшие параметры
grid_boosting = GridSearchCV(
    estimator=boosting,
    param_grid=param_grid_boosting,
    scoring="f1",
    cv=3,
    n_jobs=2,
    verbose=1
)

grid_boosting.fit(X_train, y_train)

print("Лучшие параметры:", grid_boosting.best_params_)
print("Лучший F1-score:", grid_boosting.best_score_)

Fitting 3 folds for each of 8 candidates, totalling 24 fits
Лучшие параметры: {'learning_rate': 0.05, 'max_depth': 2, 'n_estimators': 100}
Лучший F1-score: 0.8567558567558567


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import pandas as pd

# Лучшие модели после GridSearchCV
best_bagging = grid_bagging.best_estimator_
best_boosting = grid_boosting.best_estimator_

# Предсказания на тестовых данных
y_pred_bagging = best_bagging.predict(X_test)
y_pred_boosting = best_boosting.predict(X_test)

# Сравнение метрик
results = pd.DataFrame({
    "Модель": ["Bagging", "Boosting"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_bagging),
        accuracy_score(y_test, y_pred_boosting)
    ],
    "Precision": [
        precision_score(y_test, y_pred_bagging, zero_division=0),
        precision_score(y_test, y_pred_boosting, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, y_pred_bagging, zero_division=0),
        recall_score(y_test, y_pred_boosting, zero_division=0)
    ],
    "F1-score": [
        f1_score(y_test, y_pred_bagging, zero_division=0),
        f1_score(y_test, y_pred_boosting, zero_division=0)
    ]
})

display(results.round(4))


,Модель,Accuracy,Precision,Recall,F1-score
0,Bagging,0.9984,0.6944,0.8333,0.7576
1,Boosting,0.9991,0.9565,0.7333,0.8302
